# Классификация цвета автомобиля (DVM)

## Шаг 1. Данные
- Оставляем **6 самых частых** цветов по числу изображений.
- Разбиение **80% train / 20% val**

In [ ]:
from pathlib import Path
from collections import Counter

from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path(".").resolve()
DATA_ROOT = PROJECT_ROOT / "confirmed_fronts"

def colour_from_dvm_path(p: Path):
    parts = p.name.split("$$")
    if len(parts) >= 4:
        return parts[3]
    return None

def collect_images(root: Path):
    pairs = []
    for p in root.rglob("*"):
        if p.is_file() and p.suffix.lower() == ".jpg":
            colour = colour_from_dvm_path(p)
            if colour is not None:
                pairs.append((p, colour))
    return pairs

pairs = collect_images(DATA_ROOT)
print("Всего изображений:", len(pairs))

cnt = Counter(colour for _, colour in pairs)
most_common = [name for name, _ in cnt.most_common(6)]
allowed = set(most_common)
print("Топ-6 цветов:", most_common)
for name in most_common:
    print(f"  {name}: {cnt[name]}")

pairs_f = [(p, c) for p, c in pairs if c in allowed]

paths = [str(p.resolve()) for p, c in pairs_f]
labels = [most_common.index(c) for p, c in pairs_f]

train_paths, val_paths, y_train, y_val = train_test_split(
    paths,
    labels,
    test_size=0.2,
    random_state=101,
    stratify=labels,
)

print("Train:", len(train_paths), "Val:", len(val_paths))
print("Число классов:", len(most_common))
print("Пример train:", train_paths[0])

Всего изображений: 61827
Топ-6 цветов: ['Black', 'Grey', 'White', 'Blue', 'Silver', 'Red']
  Black: 14317
  Grey: 9474
  White: 9395
  Blue: 8483
  Silver: 7770
  Red: 6095
Train: 44427 Val: 11107
Число классов: 6
Пример train: /Users/moahim/PycharmProjects/kz1/confirmed_fronts/Infiniti/2011/Infiniti$$FX$$2011$$White$$37_2$$14$$image_2.jpg


## Шаг 2. Загрузка изображений

In [2]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

IMG_SIZE = 224
BATCH_SIZE = 32

imagenet_mean = (0.485, 0.456, 0.406)
imagenet_std = (0.229, 0.224, 0.225)

train_tf = transforms.Compose(
    [
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ToTensor(),
        transforms.Normalize(imagenet_mean, imagenet_std),
    ]
)

val_tf = transforms.Compose(
    [
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(imagenet_mean, imagenet_std),
    ]
)


class CarColorDataset(Dataset):
    def __init__(self, paths, labels, transform):
        self.paths = paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        x = self.transform(img)
        y = self.labels[i]
        return x, y


NUM_CLASSES = len(most_common)

train_ds = CarColorDataset(train_paths, y_train, train_tf)
val_ds = CarColorDataset(val_paths, y_val, val_tf)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0
)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

xb, yb = next(iter(train_loader))
print("батч:", xb.shape, "метки:", yb.shape)
print("число классов:", NUM_CLASSES)

батч: torch.Size([32, 3, 224, 224]) метки: torch.Size([32])
число классов: 6


## Шаг 3. Свой ResNet

- Небольшая сеть с **остаточными блоками** (`BasicBlock`), как идея ResNet.
- Оптимизатор **AdamW**, потери **CrossEntropyLoss**.

In [4]:
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import f1_score
from tqdm.auto import tqdm

device = torch.device("mps")

class BasicBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        if stride != 1 or in_ch != out_ch:
            self.skip = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride, bias=False),
                nn.BatchNorm2d(out_ch),
            )
        else:
            self.skip = nn.Identity()

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = out + self.skip(x)
        return F.relu(out)

class SmallResNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, 7, 2, 3, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(3, 2, 1),
        )
        self.layer1 = self._make_layer(64, 64, n=2, stride=1)
        self.layer2 = self._make_layer(64, 128, n=2, stride=2)
        self.layer3 = self._make_layer(128, 256, n=2, stride=2)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(256, num_classes)

    def _make_layer(self, in_ch, out_ch, n, stride):
        blocks = [BasicBlock(in_ch, out_ch, stride)]
        for _ in range(1, n):
            blocks.append(BasicBlock(out_ch, out_ch, 1))
        return nn.Sequential(*blocks)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        return self.fc(x)


def train_one_epoch(model, loader, optimizer, criterion, dev):
    model.train()
    total, seen = 0.0, 0
    for xb, yb in tqdm(loader, desc="train", leave=False):
        xb = xb.to(dev)
        yb = yb.to(dev)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total += loss.item() * xb.size(0)
        seen += xb.size(0)
    return total / max(seen, 1)

@torch.no_grad()
def eval_macro_f1(model, loader, dev):
    model.eval()
    preds, trues = [], []
    for xb, yb in loader:
        xb = xb.to(dev)
        logits = model(xb)
        pred = logits.argmax(dim=1).cpu().numpy()
        preds.extend(pred.tolist())
        trues.extend(yb.numpy().tolist())
    return f1_score(trues, preds, average="macro")


model_scratch = SmallResNet(NUM_CLASSES).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model_scratch.parameters(), lr=1e-3, weight_decay=1e-4)

EPOCHS_SCRATCH = 10
best_f1 = -1.0
best_state = None

for epoch in range(1, EPOCHS_SCRATCH + 1):
    loss_tr = train_one_epoch(
        model_scratch, train_loader, optimizer, criterion, device
    )
    f1_val = eval_macro_f1(model_scratch, val_loader, device)
    print(f"epoch {epoch:02d}, train_loss={loss_tr:.4f}, val_f1_macro={f1_val:.4f}")
    if f1_val > best_f1:
        best_f1 = f1_val
        best_state = {k: v.cpu().clone() for k, v in model_scratch.state_dict().items()}

if best_state is not None:
    model_scratch.load_state_dict(best_state)

print("SmallResNet F1 macro:", best_f1)

epoch 01, train_loss=0.7615, val_f1_macro=0.7983


epoch 02, train_loss=0.5260, val_f1_macro=0.7770


epoch 03, train_loss=0.4492, val_f1_macro=0.8622


epoch 04, train_loss=0.3931, val_f1_macro=0.7821


epoch 05, train_loss=0.3561, val_f1_macro=0.8323


epoch 06, train_loss=0.3264, val_f1_macro=0.8902


epoch 07, train_loss=0.3055, val_f1_macro=0.8977


epoch 08, train_loss=0.2822, val_f1_macro=0.8808


epoch 09, train_loss=0.2580, val_f1_macro=0.9020


epoch 10, train_loss=0.2399, val_f1_macro=0.9051
SmallResNet F1 macro: 0.9050738438953428


## Шаг 4. Предобученные ResNet-18 и MobileNet-V2 (ImageNet)

- Берём веса **IMAGENET1K_V1**, меняем последний слой на **6 классов**.
- Дообучаем теми же функциями, что и выше; **15 эпох**

In [5]:
from torchvision import models

f1_small_resnet = best_f1

def finetune_model(model, name, epochs=10, lr=3e-4):
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    crit = nn.CrossEntropyLoss()
    best = -1.0
    best_state = None
    for ep in range(1, epochs + 1):
        loss_tr = train_one_epoch(model, train_loader, opt, crit, device)
        f1v = eval_macro_f1(model, val_loader, device)
        print(f"{name}, ep {ep:02d}, loss={loss_tr:.4f}, val_f1_macro={f1v:.4f}")
        if f1v > best:
            best = f1v
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    if best_state is not None:
        model.load_state_dict(best_state)
    return best, model


rn18 = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
rn18.fc = nn.Linear(rn18.fc.in_features, NUM_CLASSES)
f1_resnet18, model_resnet18 = finetune_model(rn18, "ResNet-18")

mb = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
mb.classifier[1] = nn.Linear(mb.last_channel, NUM_CLASSES)
f1_mobilenet_v2, model_mobilenet_v2 = finetune_model(mb, "MobileNet-V2")

print("---")
print(f"SmallResNet: {f1_small_resnet:.4f}")
print(f"ResNet-18 (ImageNet): {f1_resnet18:.4f}")
print(f"MobileNet-V2 (ImageNet): {f1_mobilenet_v2:.4f}")

ResNet-18, ep 01, loss=0.3844, val_f1_macro=0.8878


ResNet-18, ep 02, loss=0.2656, val_f1_macro=0.9115


ResNet-18, ep 03, loss=0.2172, val_f1_macro=0.9198


ResNet-18, ep 04, loss=0.1854, val_f1_macro=0.9112


ResNet-18, ep 05, loss=0.1538, val_f1_macro=0.9268


...

MobileNet-V2, ep 06, loss=0.1529, val_f1_macro=0.9259


MobileNet-V2, ep 07, loss=0.1316, val_f1_macro=0.9275


MobileNet-V2, ep 08, loss=0.1196, val_f1_macro=0.9299


MobileNet-V2, ep 09, loss=0.1081, val_f1_macro=0.9245


MobileNet-V2, ep 10, loss=0.0993, val_f1_macro=0.9329
---
SmallResNet: 0.9051
ResNet-18 (ImageNet): 0.9354
MobileNet-V2 (ImageNet): 0.9329
